<a href="https://colab.research.google.com/github/olawaleaboderin/machine_learning/blob/main/Assignment2_GroundwaterPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aquifer Petrignano – Groundwater Prediction**

Comparing KFold vs TimeSeriesSplit Cross-Validation

Import necessary Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import (train_test_split, KFold,
                                     TimeSeriesSplit, cross_val_score,
                                     GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer

**----0. DATA PREPARATION -----**

Deal with missing values; build X, y chronologically; add seasonality variable

In [2]:
# Load Datasets
url = "https://raw.githubusercontent.com/olawaleaboderin/machine_learning/main/Assignment2/Petrignano.csv"
df = pd.read_csv(url)

df.head()

,Date,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
0,1/1/2009,0.0,-31.14,5.2,4.9,-24530.688,2.4
1,2/1/2009,0.0,-31.11,2.3,2.5,-28785.888,2.5
2,3/1/2009,0.0,-31.07,4.4,3.9,-25766.208,2.4
3,4/1/2009,0.0,-31.05,0.8,0.8,-27919.296,2.4
4,5/1/2009,0.0,-31.01,-1.9,-2.1,-29854.656,2.3


In [3]:
# Prompt: Convert Date column to datetime, sort chronologically, handle missing values appropriately, resample to monthly frequency.

# Remove BOM characters if present
df.columns = df.columns.str.strip().str.lstrip('\ufeff')

# Convert date and sort
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').set_index('Date')

# Handle anomalies (MODIFICATION: restricted zero replacement only to known issues)
df['Volume_C10_Petrignano'] = df['Volume_C10_Petrignano'].replace(0, np.nan)
df['Hydrometry_Fiume_Chiascio_Petrignano'] = df['Hydrometry_Fiume_Chiascio_Petrignano'].replace(0, np.nan)

# Imputation (time-based interpolation)
df = df.interpolate(method='time', limit=30, limit_direction='both')
df = df.ffill().bfill()

# Resample to monthly
monthly = df.resample('MS').mean()

In [4]:
# Prompt: Create lagged predictors (t-1, t-2), add seasonality variables, and define X (predictors) and y (response variable).
# Define target
target_col = 'Depth_to_Groundwater_P25'

# Create lag features
feature_cols = monthly.columns.tolist()
lagged_parts = []

for col in feature_cols:
    lagged_parts.append(monthly[[col]].shift(1).rename(columns={col: f'{col}_lag1'}))
    lagged_parts.append(monthly[[col]].shift(2).rename(columns={col: f'{col}_lag2'}))

X_raw = pd.concat(lagged_parts, axis=1)

# Seasonality (MODIFICATION: added cyclical encoding)
month_num = X_raw.index.month
X_raw['season_sin'] = np.sin(2 * np.pi * month_num / 12)
X_raw['season_cos'] = np.cos(2 * np.pi * month_num / 12)

# Response variable
y_raw = monthly[target_col]

# Combine and drop NaNs
data = pd.concat([X_raw, y_raw.rename('target')], axis=1).dropna()

X = data.drop(columns='target')
y = data['target']

**--- 1. HARD SPLIT ---**

In [5]:
#Prompt: Split data chronologically into training and testing sets without shuffling.

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


**--- 2. DEFINE THE TWO CV STRATEGIES ---**

In [6]:
#Prompt: Define KFold (random) and TimeSeriesSplit (temporal) strategies.

cv_naive = KFold(n_splits=5, shuffle=True, random_state=42)
cv_temporal = TimeSeriesSplit(n_splits=5)

**--- 3. EXPERIMENT WITH A FIXED MODEL---**

In [7]:
#Prompt: Build a pipeline with imputer, scaler, and DecisionTreeRegressor and compare CV strategies.

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # MODIFICATION: added imputer
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])

# Strategy 1: Naive (Shuffled)
scores_naive = cross_val_score(pipe, X_train_full, y_train_full,
                               cv=cv_naive, scoring='r2')

# Strategy 2: Temporal (Sequential)
scores_temporal = cross_val_score(pipe, X_train_full, y_train_full,
                                  cv=cv_temporal, scoring='r2')

print(f"Naive CV R2:    {scores_naive.mean():.4f}")
print(f"Temporal CV R2: {scores_temporal.mean():.4f}")

# Train on all training data and test on the unseen "future"
pipe.fit(X_train_full, y_train_full)
final_test_r2 = r2_score(y_test, pipe.predict(X_test))

print(f"\nActual Test R2 on 2021 Data: {final_test_r2:.4f}")


Naive CV R2:    0.9123
Temporal CV R2: 0.0663

Actual Test R2 on 2021 Data: -0.5067


**--- 4. MODEL SELECTION & EVALUATION ---**

In [8]:
#Prompt: Perform GridSearchCV using pipeline and evaluate performance on test data.

def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):

 # STEP A: Define the Pipeline
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])

 # STEP B: Hyperparameter grid (at least 4 different depths)
    param_grid = {
        'regressor__max_depth': [3, 5, 10, 15, 20]
    }

 # STEP C: GridSearchCV using whichever CV strategy is passed in
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring='r2',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

 # STEP D: Evaluate the best model on the truly unseen test set
    y_pred = grid.best_estimator_.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)

    print(f"\n===== Results for: {name} =====")
    print(f"Best Parameters found:       {grid.best_params_}")
    print(f"Internal CV Score   (R²):    {grid.best_score_:.4f}")
    print(f"Independent Test Score (R²): {test_r2:.4f}")
    print(f"Gap (CV − Test):             {grid.best_score_ - test_r2:.4f}")

    return grid

**--- 5. RUNNING THE COMPARISON ---**

In [13]:
# prompt: Compare Naive KFold and TimeSeriesSplit model selection results.

result_naive = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test, cv_naive, "Naive K-Fold"
)

result_temporal = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test, cv_temporal, "TimeSeriesSplit"
)


===== Results for: Naive K-Fold =====
Best Parameters found:       {'regressor__max_depth': 10}
Internal CV Score   (R²):    0.9123
Independent Test Score (R²): -0.5067
Gap (CV − Test):             1.4190

===== Results for: TimeSeriesSplit =====
Best Parameters found:       {'regressor__max_depth': 15}
Internal CV Score   (R²):    0.0703
Independent Test Score (R²): -0.5067
Gap (CV − Test):             0.5770
